# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.


In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.


In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'related br

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano


In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-9B
Updated
7 days ago
•
868k
•
590
Lightricks/LTX-2.3
Updated
3 days ago
•
175k
•
346
Qwen/Qwen3.5-35B-A3B
Updated
9 days ago
•
1.14M
•
1.04k
Qwen/Qwen3.5-0.8B
Updated
6 days ago
•
406k
•
317
Qwen/Qwen3.5-4B
Updated
7 days ago
•
349k
•
294
Browse 2M+ models
Spaces
Running
on
Zero
Featured
428
Omni Video Factory
🏆
428
text to video, image to video, video extend
Running
on
Zero
145
OBLITERATUS
💥
145
One-click model liberation + chat playground
Running
on
Zero
MCP
Featured
153
FireRed Image Edit 1.0 Fast
🌖
153
FireRed-Image

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n7 days ago\n•\n868k\n•\n590\nLightricks/LTX-2.3\nUpdated\n3 days ago\n•\n175k\n•\n346\nQwen/Qwen3.5-35B-A3B\nUpdated\n9 days ago\n•\n1.14M\n•\n1.04k\nQwen/Qwen3.5-0.8B\nUpdated\n6 days ago\n•\n406k\n•\n317\nQwen/Qwen3.5-4B\nUpdated\n7 days ago\n•\n349k\n•\n294\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n428\nOmni Video Factory\n🏆\n428\ntext to video, i

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

## About Hugging Face  
Hugging Face is the leading AI community platform dedicated to building the future of machine learning through open collaboration. It serves as a central hub where machine learning engineers, scientists, and enthusiasts can share, explore, and experiment with an extensive repository of open-source AI models, datasets, and applications. The platform fosters an open, ethical, and inclusive AI ecosystem by empowering the next generation of AI practitioners to learn, contribute, and innovate together.

## What We Offer  
- **2 Million+ Models:** Access a vast and diverse catalog of machine learning models across multiple modalities including text, image, video, audio, and even 3D. Popular models like Qwen (versions ranging from 0.8B to 35B parameters) and specialized models such as Lightricks/LTX are actively maintained and frequently updated.  
- **500,000+ Datasets:** Discover and contribute to a rich collection of datasets to fuel your AI experiments and training needs, covering various domains and complexities.  
- **Spaces:** Run and host interactive ML apps easily with a range of featured projects such as video generation tools, image editing, and chat playgrounds, enabling instant model deployment with zero infrastructure hassle.  
- **Community & Collaboration:** Engage with a fast-growing global community of AI researchers and practitioners. Share your work publicly, build your professional ML portfolio, and collaborate openly on projects.  
- **Enterprise Solutions:** For businesses and teams looking to accelerate their AI initiatives, Hugging Face provides paid compute resources and enterprise-grade offerings to scale projects efficiently.  

## Company Culture  
Hugging Face fosters a collaborative and inclusive environment passionate about open-source innovation and ethical AI development. The platform is built on transparency and community-driven progress, with a strong emphasis on education and knowledge sharing. This culture inspires continuous learning, collective problem-solving, and creativity — shaping the AI tools and standards of tomorrow.

## Our Customers and Users  
Hugging Face supports a wide ecosystem of users including:  
- Individual ML engineers and data scientists  
- Academic and research institutions  
- AI startups and innovating tech companies  
- Large enterprises integrating AI models into products and workflows

With millions of monthly users accessing models and datasets, Hugging Face is trusted globally as the go-to community platform for AI development and deployment.

## Careers & Opportunities  
Join Hugging Face if you want to contribute at the forefront of open-source AI. The company welcomes talented individuals eager to work in a dynamic, community-first environment focused on cutting-edge technology. Careers opportunities include roles in machine learning research, software engineering, community management, and enterprise solutions.

Discover and build your future with AI alongside a passionate community at Hugging Face.

---

**Connect with Us**  
- Explore our models and datasets on the Hugging Face Hub  
- Launch your ML app with Spaces  
- Join the community to collaborate and learn  
- Visit [huggingface.co](https://huggingface.co) and sign up today!

*Together, let's build the future of AI.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation


In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the premier AI community and collaboration platform dedicated to building the future of machine learning (ML). As a hub where ML engineers, scientists, and end users come together, Hugging Face enables sharing, discovering, and experimenting with open-source machine learning models, datasets, and applications. The company empowers the next generation of AI practitioners to learn, collaborate, and innovate in an open and ethical AI ecosystem.

With over **2 million models**, **500k+ datasets**, and **1 million+ AI applications**, Hugging Face stands at the forefront of the AI revolution, supporting all modalities including text, image, video, audio, and even 3D data. Their fast-growing community and cutting-edge science team continuously push the boundaries of technology.

---

## Platform Highlights

- **Explore 2M+ Models:** Access and contribute to a vast library of machine learning models updated regularly by a global community.
- **Host & Collaborate:** Unlimited public hosting of models, datasets, and applications to accelerate research and deployment.
- **Spaces:** Launch machine learning applications in a simple, shareable format powered by advanced compute like ZeroGPU.
- **Datasets:** Browse over 500,000 curated datasets for diverse AI training needs.
- **Multi-Modal AI:** Support for building applications across various data types — text, images, video, audio, and 3D.
- **Open Source Stack:** Powerful tools and libraries open to all, accelerating innovation in machine learning.
- **Community-Driven:** Share your work, build your ML portfolio, and engage with other AI developers worldwide.

---

## For Businesses & Enterprises

Hugging Face offers **Team & Enterprise Plans** designed to scale AI initiatives with:

- Enterprise-grade security and granular access control (SSO, audit logs, resource groups).
- Flexible contract options tailored to organizational needs.
- Dedicated support and analytics dashboards to monitor repository usage.
- Advanced compute options including multiplied ZeroGPU quotas for enhanced performance.
- Private storage with scalable add-ons.
- Centralized token management and billing for inference providers.

Whether a startup or a large corporation, Hugging Face provides a secure, scalable platform for building, deploying, and managing AI models in production environments.

---

## Company Culture

At its core, Hugging Face is a **collaborative, community-first organization** dedicated to ethical AI development and open science. The company prides itself on:

- Fostering an open and inclusive environment where innovation thrives.
- Empowering users and contributors from all backgrounds to participate meaningfully in AI advancement.
- Emphasizing transparency, accessibility, and responsible AI usage.
- Bringing together a talented science and engineering team passionate about pushing AI boundaries.

---

## Careers at Hugging Face

Join a vibrant, mission-driven team at the frontier of AI technology. Hugging Face offers exciting opportunities in:

- Machine Learning Engineering
- Research Science
- Software Development
- Community Management
- Enterprise Solutions and Support

Working at Hugging Face means contributing to a leading open-source platform that shapes the future of AI and working alongside passionate experts in a dynamic, innovative environment.

Explore current openings and apply to be part of the AI revolution at Hugging Face.

---

## Connect with Hugging Face

- Website: [huggingface.co](https://huggingface.co)
- GitHub, Twitter, LinkedIn, Discord: Links available on the website
- Engage with an extensive and engaged community contributing to the future of machine learning.

---

### Brand Essence

- **Colors:** #FFD21E (yellow), #FF9D00 (orange), #6B7280 (gray)
- **Logo:** Available in multiple formats for brand use

---

Hugging Face is not just a company; it’s **the movement building the future of AI together.**  

Unlock the power of machine learning with Hugging Face — your partner in AI innovation.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
</td>
</tr>

</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>
